In [1]:
import pandas as pd
from scipy.special import comb
from scipy.stats import ttest_ind
from statsmodels.stats.proportion import proportions_ztest
import ast
import math
import numpy as np

In [2]:
def prob(sequence):
    return sum(sequence) / len(sequence)

def split(sequence):

    n1 = math.ceil(len(sequence) / 2)

    s1 = sequence[:n1] # includes the first n1 elements
    s2 = sequence[n1:] # the rest of the elements

    return s1,s2

### Returns Model 1 metric. Sequence is type list

## Question: Are we using the entire sequence's probability of success or the second half?

def c1(sequence):
    _, s2 = split(sequence) 
    n2 = len(s2) # Number of games in second half
    k2 = sum(s2) # Number of wins in second half
    p = prob(sequence)

    sum_ = 0
    for i in range(k2 + 1):
        sum_ += comb(n2, i) * p**i * (1-p)**(n2-i)

    return 1 / sum_ if sum_ > 0 else float('inf')

### Returns Model 2 metric. Sequence is type list. 
    

def c2(sequence):
    s1, s2 = split(sequence)

    k1 = sum(s1)
    k2 = sum(s2)

    n1 = len(s1)
    n2 = len(s2)

    _, p = proportions_ztest([k1, k2], [n1, n2], alternative = "larger")

    if (p == 0):
        return math.inf
    else:
        return 1 / p

    


### Returns Model 3 metric. Sequence is type list. N is type int. 

def c3(sequence, N):

    sequence = np.asarray(sequence)   # convert once
    M = 0

    _, s2 = split(sequence)
    k2 = np.sum(s2)

    for _ in range(N):
        shuffled = np.random.permutation(sequence)  # new shuffled copy
        _, tmp = split(shuffled)

        if np.sum(tmp) <= k2:
            M += 1

    return N / M if M != 0 else math.inf

In [3]:
soccer = pd.read_csv("../data/processed/soccer_binary.csv")
us = pd.read_csv("../data/processed/us_leagues.csv")

# Get list equivalent instead of string

soccer['Sequence_Numeric'] = soccer['Sequence'].apply(ast.literal_eval)
us['Sequence_Numeric'] = us['Sequence'].apply(ast.literal_eval)

In [4]:
## Add Columns n, n1, n2, k, k1, k2

# total length of each sequence
us['n'] = us['Sequence_Numeric'].apply(len)

# split once per row
splits = us['Sequence_Numeric'].apply(split)

# lengths
us['n1'] = splits.apply(lambda x: len(x[0]))
us['n2'] = splits.apply(lambda x: len(x[1]))

# sums
us['k'] = us['Sequence_Numeric'].apply(sum)
us['k1'] = splits.apply(lambda x: sum(x[0]))
us['k2'] = splits.apply(lambda x: sum(x[1]))

# ---- #

soccer['n'] = soccer['Sequence_Numeric'].apply(len)

splits = soccer['Sequence_Numeric'].apply(split)

soccer['n1'] = splits.apply(lambda x: len(x[0]))
soccer['n2'] = splits.apply(lambda x: len(x[1]))

soccer['k'] = soccer['Sequence_Numeric'].apply(sum)
soccer['k1'] = splits.apply(lambda x: sum(x[0]))
soccer['k2'] = splits.apply(lambda x: sum(x[1]))

In [5]:
# Apply C1

soccer['c1'] = [c1(seq) for seq in soccer['Sequence_Numeric']]
us['c1'] = [c1(seq) for seq in us['Sequence_Numeric']]

In [6]:
# Apply C2

soccer['c2'] = [c2(seq) for seq in soccer['Sequence_Numeric']]
us['c2'] = [c2(seq) for seq in us['Sequence_Numeric']]

C:\Users\ryanj\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\stats\weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std


In [7]:
N = 1000

soccer['c3'] = [c3(seq, N) for seq in soccer['Sequence_Numeric']]
us['c3'] = [c3(seq, N) for seq in us['Sequence_Numeric']]

In [8]:
us.columns

Index(['League', 'Season', 'Team', 'Sequence', 'Sequence_Numeric', 'n', 'n1',
       'n2', 'k', 'k1', 'k2', 'c1', 'c2', 'c3'],
      dtype='object')

In [9]:
## Recalculate top 100 in c3 column 

us_indexes = list(us.sort_values('c3', ascending=False).head(100).index)
soccer_indexes = list(soccer.sort_values('c3', ascending = False).head(100).index)

N = 100000

mask = us.index.isin(us_indexes)
us.loc[mask, 'c3'] = us.loc[mask, 'Sequence_Numeric'].apply(lambda seq: c3(seq, N))

mask = soccer.index.isin(soccer_indexes)
soccer.loc[mask, 'c3'] = soccer.loc[mask, 'Sequence_Numeric'].apply(lambda seq: c3(seq, N))

In [10]:
us_out = us[['League', 'Season', 'Team', 'Sequence', 'n', 'n1', 'n2', 'k', 'k1', 'k2', 'c1', 'c2', 'c3']]
soccer_out = soccer[['League', 'Season', 'Team', 'Sequence', 'n', 'n1', 'n2', 'k', 'k1', 'k2', 'c1', 'c2', 'c3']]

soccer_output = "../output/csvs/soccer_projII.csv"
us_output = "../output/csvs/us_leagues_projII.csv"

soccer_out.to_csv(soccer_output)
us_out.to_csv(us_output)